In [1]:
"""
Generador de demanda vehicular estocástica (Poisson) para SUMO
"""
import numpy as np
import xml.etree.ElementTree as ET
from xml.dom import minidom

In [2]:
# ─────────────────────────────────────────────────────────────
# 1. PARAMETROS GENERALES
# ─────────────────────────────────────────────────────────────
SEED      = 42    # fija la reproducibilidad: misma seed -> mismo .rou.xml siempre
DURACION  = 3600   # 1 hora de simulacion = ventana de "hora pico" definida en la tesis
INTERVALO = 60     # se re-calcula la tasa Poisson cada 60s (resolucion de muestreo)

# Flujo base obtenido del EDA propio sobre 7 muestras de video del acceso
# Oeste (metodologia Sarango, 2025).
#   promedio: 1205 veh/h -> 20.0833 veh/min  (usado como LAMBDA_MIN_BASE)
#   pico:     1291 veh/h -> 21.5167 veh/min  (referencia, no se usa en Poisson)
#   valle:    1125 veh/h -> 18.7500 veh/min  (referencia, no se usa en Poisson)
# NOTA IMPORTANTE: Poisson solo admite UNA tasa constante (lambda), por eso
# aqui se usa el PROMEDIO. El pico y el valle del EDA no se usan en este
# script porque Poisson no modela fluctuacion temporal -- para eso esta el
# generador Weibull, que sí usa pico/valle a traves de la forma de la curva.
LAMBDA_MIN_BASE = 1205 / 60  # ≈ 20.0833 veh/min

# Reparto asimetrico del flujo total entre los dos accesos modelados.
# Justificacion: Sarango solo aforo el acceso Oeste (Av. Isidro Ayora), que es
# la via de mayor jerarquia de la zona. No existen datos de campo para el
# acceso Norte (Av. 8 de Diciembre), por lo que se asume un reparto 60/40
# representativo de la hora pico matutina, donde el flujo principal viaja
# desde el oeste (sector residencial/terminal terrestre) hacia el centro.
SPLIT_OESTE = 0.60
SPLIT_NORTE = 0.40

LAMBDA_OESTE = LAMBDA_MIN_BASE * SPLIT_OESTE  # ≈ 11.69 veh/min
LAMBDA_NORTE = LAMBDA_MIN_BASE * SPLIT_NORTE  # ≈  7.79 veh/min


In [3]:
# ─────────────────────────────────────────────────────────────
# 2. ACCESOS Y RUTAS
# ─────────────────────────────────────────────────────────────
# Cada acceso tiene su propio lambda (tasa Poisson independiente).
# Los porcentajes de giro (0.70 / 0.30) estan DENTRO de cada acceso,
# es decir, del 100% de autos que entran por Oeste, el 70% sigue recto.
# Estos splits son un supuesto de ingenieria de trafico estandar
# (movimiento recto > giro) ya que Sarango no detalla el reparto por giro.
ACCESOS = [
    {
        "nombre": "Oeste",
        "lambda_min": LAMBDA_OESTE,
        "rutas": [
            ("ruta_OE", "-E4", "E3", 0.70),  # Oeste->Este: recto (movimiento principal)
            ("ruta_OS", "-E4", "E6", 0.30),  # Oeste->Sur:  giro derecha
        ],
    },
    {
        "nombre": "Norte",
        "lambda_min": LAMBDA_NORTE,
        "rutas": [
            ("ruta_NS", "-E5", "E6", 0.70),  # Norte->Sur:  recto (movimiento principal)
            ("ruta_NE", "-E5", "E3", 0.30),  # Norte->Este: giro izquierda
        ],
    },
]

# Composicion vehicular tomada de la distribucion de clases reportada
# por Sarango (2025) en su conteo con YOLOv5: autos, motos y buses/pesados.
conteo_car  = 875   # clase "Auto-suv-pickup"
conteo_moto = 175   # clase "moto"
conteo_bus  = 116   # clase "V.-Pesado"
total_tipos = conteo_car + conteo_moto + conteo_bus

TIPOS = [
    ("car",  "passenger",  round(conteo_car  / total_tipos, 4)),  # ≈ 0.7505
    ("moto", "motorcycle", round(conteo_moto / total_tipos, 4)),  # ≈ 0.1501
    ("bus",  "bus",        round(conteo_bus  / total_tipos, 4)),  # ≈ 0.0995
]

In [4]:
# ─────────────────────────────────────────────────────────────
# 3. GENERACION ESTOCASTICA — POISSON (tasa constante)
# ─────────────────────────────────────────────────────────────
def generar_tiempos_acceso(lambda_min, seed_offset):
    """
    Genera tiempos de llegada con un proceso de Poisson homogeneo.
    En cada intervalo de 60s se sortea n ~ Poisson(lambda_min), y esos
    n vehiculos se distribuyen con tiempos UNIFORMES dentro del intervalo
    (no hay preferencia temporal: la tasa no cambia con t).
    """
    rng = np.random.default_rng(SEED + seed_offset)  # seed distinta por acceso -> independencia estadistica
    tiempos = []
    for t0 in range(0, DURACION, INTERVALO):
        n = rng.poisson(lambda_min)               # nro. de llegadas en este minuto
        offsets = rng.uniform(0, INTERVALO, size=n)  # instante exacto dentro del minuto
        tiempos.extend(t0 + offsets)
    tiempos.sort()
    return tiempos

In [5]:
# ─────────────────────────────────────────────────────────────
# 4. CONSTRUCCION DEL XML
# ─────────────────────────────────────────────────────────────
def construir_xml(tiempos_por_acceso):
    root = ET.Element("routes")
    root.set("xmlns:xsi", "http://www.w3.org/2001/XMLSchema-instance")
    root.set("xsi:noNamespaceSchemaLocation",
             "http://sumo.dlr.de/xsd/routes_file.xsd")

    # vTypes con modelo Krauss (estandar SUMO para car-following).
    # sigma = "imperfeccion del conductor": 0 -> robot perfecto, 1 -> muy erratico.
    # Se sube a 0.8/0.9 (en vez del 0.5 por defecto) porque Sarango documenta
    # que la falta de senaletica en el tramo genera entre 2 y 4 columnas
    # desordenadas de vehiculos -> se necesita mas "ruido" de comportamiento
    # para que el modelo refleje ese desorden real, no un trafico ideal.
    ET.SubElement(root, "vType", id="car",  vClass="passenger",
                  accel="3.0", decel="4.5", sigma="0.8",
                  length="5.0", minGap="2.5", maxSpeed="13.9",   # 13.9 m/s = 50 km/h, igual al limite del .net.xml
                  carFollowModel="Krauss", color="0.8,0.8,0.8")
    ET.SubElement(root, "vType", id="moto", vClass="motorcycle",
                  accel="3.5", decel="5.0", sigma="0.9",          # motos aun mas erraticas (zigzag tipico en Loja)
                  length="2.2", minGap="2.0", maxSpeed="16.7",    # motos algo mas rapidas (60 km/h aprox.)
                  carFollowModel="Krauss", color="0.9,0.5,0.1")
    ET.SubElement(root, "vType", id="bus",  vClass="bus",
                  accel="2.0", decel="4.5", sigma="0.3",          # buses mas conservadores/predecibles
                  length="12.0", minGap="2.5", maxSpeed="11.1",   # 11.1 m/s = 40 km/h (vehiculo pesado, mas lento)
                  carFollowModel="Krauss", color="0.2,0.6,0.2")

    # Definicion de rutas (edges de entrada/salida segun connections del .net.xml)
    for acceso in ACCESOS:
        for rid, e_from, e_to, _ in acceso["rutas"]:
            ET.SubElement(root, "route", id=rid, edges=f"{e_from} {e_to}")

    # Mezcla de vehiculos de ambos accesos, ordenados por tiempo de salida
    # (SUMO requiere que los <vehicle> aparezcan en orden creciente de depart)
    todos = []
    for acceso, (tiempos, seed_offset) in zip(ACCESOS, tiempos_por_acceso):
        rng    = np.random.default_rng(SEED + seed_offset)
        rutas  = acceso["rutas"]
        p_r    = np.array([r[3] for r in rutas]); p_r /= p_r.sum()  # normaliza probabilidades de giro
        p_t    = np.array([v[2] for v in TIPOS]);  p_t /= p_t.sum()  # normaliza probabilidades de tipo
        idx_r  = rng.choice(len(rutas), size=len(tiempos), p=p_r)
        idx_t  = rng.choice(len(TIPOS), size=len(tiempos), p=p_t)
        for t, ir, it in zip(tiempos, idx_r, idx_t):
            todos.append({
                "depart": t,
                "type":   TIPOS[it][0],
                "route":  rutas[ir][0],
            })

    todos.sort(key=lambda x: x["depart"])

    for i, v in enumerate(todos):
        ET.SubElement(root, "vehicle",
                      id=f"veh_{i}",
                      type=v["type"],
                      route=v["route"],
                      depart=f"{v['depart']:.2f}",
                      departLane="best",     # SUMO elige el carril menos congestionado al entrar
                      departSpeed="0")       # arranque desde parado (mas realista para una via urbana)
    return root, len(todos)


In [6]:
# ─────────────────────────────────────────────────────────────
# 5. GUARDAR Y REPORTE
# ─────────────────────────────────────────────────────────────
def guardar(root, filepath):
    """Formatea el arbol XML con sangria y lo guarda en disco."""
    xml_str = ET.tostring(root, encoding="utf-8")
    parsed_str = minidom.parseString(xml_str)
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(parsed_str.toprettyxml(indent="    "))

def reporte(accesos_data):
    total = sum(len(t) for t, _ in accesos_data)
    print("=" * 56)
    print("  REPORTE — Demanda Poisson (tasa constante)")
    print("=" * 56)
    for acceso, (tiempos, _) in zip(ACCESOS, accesos_data):
        print(f"  Acceso {acceso['nombre']}")
        print(f"    lambda objetivo: {acceso['lambda_min']:.2f} veh/min")
        print(f"    vehiculos generados: {len(tiempos)}")
        print(f"    tasa real obtenida:  {len(tiempos)/DURACION*60:.2f} veh/min")
        print()
    print(f"  Total interseccion: {total} veh")
    print(f"  Ratio real Oeste:Norte = "
          f"{len(accesos_data[0][0])/len(accesos_data[1][0]):.2f}:1  "
          f"(objetivo: {SPLIT_OESTE/SPLIT_NORTE:.2f}:1)")
    print("=" * 56)

In [9]:
print("Generando demanda con distribucion Poisson...\n")

tiempos_oeste = generar_tiempos_acceso(LAMBDA_OESTE, seed_offset=0)
tiempos_norte = generar_tiempos_acceso(LAMBDA_NORTE, seed_offset=10)

accesos_data = [
    (tiempos_oeste, 1),   # seed_offset+1 usado en construir_xml para elegir ruta/tipo
    (tiempos_norte, 11),
]

root, total = construir_xml(accesos_data)

out = r"./loja_intersection_poisson.rou.xml"
guardar(root, out)
reporte(accesos_data)
print(f"\n✓ Archivo listo: {out}")

Generando demanda con distribucion Poisson...

  REPORTE — Demanda Poisson (tasa constante)
  Acceso Oeste
    lambda objetivo: 12.05 veh/min
    vehiculos generados: 711
    tasa real obtenida:  11.85 veh/min

  Acceso Norte
    lambda objetivo: 8.03 veh/min
    vehiculos generados: 468
    tasa real obtenida:  7.80 veh/min

  Total interseccion: 1179 veh
  Ratio real Oeste:Norte = 1.52:1  (objetivo: 1.50:1)

✓ Archivo listo: ./loja_intersection_poisson.rou.xml
